# CARI-rCARI Validation study - Iraq - Phase 1
This notebook calculate basic food security indicators and prepared the datasets for the four versions of the CARI-rCARI surveys in Iraq (F2F version A, F2F version B, Remote A and Remote B) for visualization in Tableau.

In [29]:
%load_ext autoreload
# import required libraries
import os
import pandas as pd
import re
import numpy as np
from datetime import datetime
# imports the functions to compute indicators
import indicators
# import constants (uuids of households)
import constants

#  Pandas Dataframe display options 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data
### Import dataset
Datasets are imported from the parent folder ```raw```. The function gets all csv files in that folder.

In [30]:
# import datasets
parent = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
data_path = "raw/all"
path = os.path.join(parent, data_path)
helper_file = f"{parent}\\raw\\helper.xlsx"

def get_datasets(path):
    """Get list of datasets from given path"""
    return [pd.read_csv(os.path.join(path, f)) for f in os.listdir(path) if f.endswith(".csv")]


### Import helper file
The helper file contains the column and labels for each of the surveys. Will be joined with the four datasets later, so the final files have the standardized column names and labels

In [31]:
def get_helper(file):
    lst = []
    helper_list = ["helper_F2F_A", "helper_F2F_B", "helper_R_A", "helper_R_B"]

    for sheet in helper_list:
        df = pd.read_excel(file, sheet_name=sheet)
        lst.append(df)
    return lst


In [32]:
# data is a list of pandas dataframes, each corresponding to the four surveys (F2F A, F2F B, Remote A, Remote B)
data = get_datasets(path)
# helpers is a pandas dataframe, with  a file with labels and variables names
helpers = get_helper(helper_file)

## Cleaning data

### Clean and standardize columns

In this code block, each of the surveys in the data is standardized: each datasets get the same columns so they can be unioned later; dates are parsed; and a column to calculate the length of the interview is created.

In [33]:
surveys = {
    0: "F2F A",
    1: "F2F B",
    2: "Remote A",
    3: "Remote B"
}

for i, df in enumerate(data):
    # Add questionnaire name as column
    df["Questionnaire"] = surveys[i]
    # parse dates
    df['start'] = pd.to_datetime(df['start'])
    df['end'] = pd.to_datetime(df['end'])
    # calculate interview_lenght
    df['interview_length'] = round(
        abs((df.start - df.end) / pd.Timedelta(minutes=1)), 1)
    # create column in all datasets, if not exist it populates it with NaN
    # this is done so the four datasets can be unioned later
    df['ADMIN1Name'] = df.get('ADMIN1Name', np.nan)
    df["HHStatus"] = df.get("HHStatus", np.nan)
    df["RESPName"] = df.get("RESPName", np.nan)
    # cast HHID as string
    df['HHID'] = df['HHID'].astype(str)

### Rename columns

This block corrects the naming of a column (LcsEN_em_ChildWork) and removes all rows where respondant was not available or didn't complete the remote survey.

In [34]:
f2fa, f2fb, ra, rb = data[:4]

for df in [ra, rb]:

    df.rename(
        columns={'LcsEN_em_ChildWork': 'Lcs_em_ChildWork'}, inplace=True)
    # drops column where RESPAvailable or RESPAge are NAN
    df.dropna(subset=['RESPAvailable', 'RESPAge'], inplace=True)

# Check if in dataset still has values for RESPAvailable that are not 1 (meaning that the respondant was NOT available)
mask = ~rb['RESPAvailable'].isin([1])
df_filtered = rb[mask]
# this checks if the dataset is empty (as it should!)
df_filtered.head()


,start,end,today,deviceid,SvyDate,EnuName,EnuSex,HHID,HHPhNmb,HHIntFirstSec,ADMIN5Name,RESPpart,RESPpart_oth,RESPAvailable,RESPcallback,RESPAge,RESPvoice,RESPname,RESPSex,HHSize,RESPRelationHHH,RESPRelationHHH_oth,FCSStap,FCSPulse,FCSDairy,FCSPr,FCSVeg,FCSFruit,FCSFat,FCSSugar,rCSILessQlty,rCSIBorrow,rCSIMealSize,rCSIGenderMealSize,rCSIMealAdult,rCSIGenderMealAdult,rCSIMealNb,rCSIGenderMealNb,Lcs_stress_DomAsset,Lcs_stress_CrdtFood,Lcs_stress_Saving,Lcs_stress_BorrowCash,Lcs_crisis_ProdAssets,Lcs_crisis_HealthEdu,Lcs_crisis_OutSchool,Lcs_em_ChildWork,Lcs_em_Begged,Lcs_em_IllegalAct,HHExpF_MNCRD_7D,HHExpF_GiftAid_7D,HHExpF_Own_7D,HHExpNF_MNCRD_1M,HHExpNF_GiftAid_1M,HHIncNb,HHIncFirst_SRi,HHIncFirst_Oth,HHIncChg,followup_visit,RESPPhNmbAlt,comments,instanceID,_id,_uuid,_submission_time,_date_modified,_tags,_notes,_version,_duration,_submitted_by,_total_media,_media_count,_media_all_received,_xform_id,Questionnaire,interview_length,ADMIN1Name,HHStatus,RESPName


#### Drop columns not needed in the visualization

In [35]:
to_drop = [col for col in f2fa.columns if re.search(
    r'(MDDI|HHPreg|PWMDDW|PCMAD)', col)]
f2fa.drop(to_drop, axis=1, inplace=True)

#### Correct HHIDs

Correct HHIDs in Remote A

In [36]:
correct_hhids = {
    "7704647214":"320101006",
"7704987889": "320101002",
"7706230392": "320101001",
"7724960495": "320101005",
"7731467223": "320101004",
"774648295":"320101007"}

ra.replace({"HHID": correct_hhids}, inplace=True)

Remove duplicate rows in Remote A

In [37]:
ra = ra.loc[~ra['_uuid'].isin(constants.REMOTE_A_UUIDS_TO_DROP)]

Correct HHIDs in F2F

In [38]:
def update_hhid_from_uuid(dataframe, uuid_col, hhid_col, uuid_list, correct_hhid_list):
    """
    Update the hhid column of the dataframe based on the values in the uuid column.
    The correct hhids are obtained from the correct_hhid_list which is mapped to the 
    uuids from the uuid_list.

    Parameters:
        dataframe (pandas DataFrame): The dataframe to update
        uuid_col (str): The name of the uuid column
        hhid_col (str): The name of the hhid column
        uuid_list (list): The list of uuids to match
        correct_hhid_list (list): The corresponding correct hhids

    Returns:
        None
    """
    dataframe.loc[dataframe[uuid_col].isin(uuid_list), hhid_col] = dataframe.loc[dataframe[uuid_col].isin(
        uuid_list), uuid_col].map(dict(zip(uuid_list, correct_hhid_list), default=None))
    return dataframe


f2fa = update_hhid_from_uuid(f2fa, '_uuid', 'HHID', constants.F2FA_UUIDS, constants.F2FA_CORRECT_HHIDS)
f2fb = update_hhid_from_uuid(f2fb, '_uuid', 'HHID', constants.F2FB_UUIDS, constants.F2FB_CORRECT_HHIDS)
    
# # Check if values exist
# mask = f2fb["HHID"].isin(constants.F2FB_CORRECT_HHIDS)
# df_filtered = f2fb[mask]


In [39]:
len_corrected_remote_a = len(correct_hhids)
len_corrected_f2fa = len(constants.F2FB_UUIDS)
len_corrected_f2fb = len(constants.F2FA_UUIDS)

print(f"{len_corrected_remote_a + len_corrected_f2fa + len_corrected_f2fb} values corrected")


49 values corrected


### Duplicate counts

This code block produces a CSV file counting the duplicate HHIDs per survey.

In [40]:
duplicate_counts = pd.DataFrame([], columns=["survey", "unique_count", "total"])

for i, df in enumerate(data):
    unique = len(pd.unique(df['HHID']))
    total = len(df.index)
    new_row = pd.DataFrame(
        {'survey': surveys[i], 'unique_count': unique, 'total': total}, index=[i])
    duplicate_counts = pd.concat([duplicate_counts, new_row])

duplicate_counts.to_csv(f"{parent}/clean/duplicate_counts.csv")


In [48]:
duplicate_counts

,survey,unique_count,total
0,F2F A,300,305
1,F2F B,284,285
2,Remote A,298,316
3,Remote B,244,248


## Indicators

In [41]:
data = indicators.food_consumption_nutrition_rCSI(data)
data = indicators.essential_needs_coping(data)
data = indicators.food_security_coping(data)
# data = apply_cari_income(data)

## Reshape

Pivot datasets from wide to long format

In [42]:
def select_cols(df):
    to_keep = ["_id", "HHStatus", 'start', 'end', 'today', "deviceid", "SvyDate", 'ADMIN1Name',
               'ADMIN5Name', 'EnuName', 'EnuSex', 'HHID', 'RESPAge', 'RESPName', 'RESPSex', 'interview_length', "Questionnaire"]
    to_pivot = df.columns.difference(to_keep)
    return (to_keep, to_pivot)


def pivot_df(data): return [pd.melt(df, id_vars=select_cols(df)[
    0], value_vars=select_cols(df)[1]) for df in data]


In [43]:
data = pivot_df(data)

### Joining datasets

#### Join with Helper file
The helper file contains the labels for the questions

In [44]:
for df in data:
    df.rename(
        columns={'variable': 'Question', 'value': 'Value'}, inplace=True)

Left join between dataset and helper file - commented out as currently not used in the dashboard.

In [45]:
# def merge_dataframes(first, second, on=None, how='left'):
#     out = []
#     for l, r in zip(first, second):
#         result = pd.merge(l, r, on=on, how=how)
#         out.append(result)
#     return out


# data = merge_dataframes(data, helpers, on='question', how='left')

In [46]:
f2fa, f2fb, ra, rb = data[:4]
# clean unwanted Unname column
ra.drop(ra.filter(regex="Unname"),axis=1, inplace=True)


## Export and save

Save the four datasets in CSV: the two remotes are together, the F2F A and F2F B are saved as separate files

In [47]:
pd.concat([ra, rb]).to_csv(f"{parent}/clean/remote_a_b.csv")
f2fa.to_csv(f"{parent}/clean/f2fa.csv")
f2fb.to_csv(f"{parent}/clean/f2fb.csv")
